# Phase 3 DAEAC: Class-Balanced Focal Loss

This notebook runs Phase 3 only: class-balanced focal loss for source supervised classification. It keeps real RR rows, keeps Phase 2 reliable pseudo-labeling off, and does not enable dynamic weighting, task alignment, FCBA, Transformer, MCC, or Gram-OT.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_INPUT = Path('/kaggle/input')
CONFIG = 'configs/phase6_daeac_paper_phase3_focal.yaml'
BASE_PREFIX = 'daeac_base_phase3_focal'
ADAPT_PREFIX = 'daeac_phase3_focal'
OUTPUT_DIR = Path('outputs/phase6_daeac_paper_phase3_focal')

# If the repo is already the notebook working directory, keep it. Otherwise set REPO_DIR manually.
cwd = Path.cwd()
if (cwd / 'scripts/phase6_daeac_paper').exists():
    REPO_DIR = cwd
elif (cwd / 'ecg_thesis' / 'scripts/phase6_daeac_paper').exists():
    REPO_DIR = cwd / 'ecg_thesis'
elif (KAGGLE_WORKING / 'Best-thesis-in-council' / 'ecg_thesis').exists():
    REPO_DIR = KAGGLE_WORKING / 'Best-thesis-in-council' / 'ecg_thesis'
else:
    candidates = [p for p in KAGGLE_INPUT.glob('**/ecg_thesis') if (p / 'scripts/phase6_daeac_paper').exists()]
    if not candidates:
        raise FileNotFoundError('Could not find ecg_thesis. Attach the repo as a Kaggle dataset or set REPO_DIR manually.')
    source_repo = candidates[0]
    dest = KAGGLE_WORKING / 'Best-thesis-in-council' / 'ecg_thesis'
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        shutil.copytree(source_repo, dest)
    REPO_DIR = dest

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('REPO_DIR =', REPO_DIR)
print('CONFIG =', CONFIG)

In [ ]:
REQ = REPO_DIR / 'requirements.txt'
if REQ.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=True)
else:
    print('requirements.txt not found; using Kaggle environment packages')

## Attach Data And Optional Checkpoints

Attach a Kaggle dataset containing the Phase 6 record split `.npz` files. For full Phase 3, the notebook trains a focal source base checkpoint first. If you already have one, attach it and the cell below will copy it when found.

In [ ]:
required_npz = ['ds1_train.npz', 'ds1_val.npz', 'ds2_train.npz', 'ds2_val.npz', 'ds2_test.npz']
data_dest = REPO_DIR / 'data/processed/phase6_daeac_record_splits'
data_dest.mkdir(parents=True, exist_ok=True)

candidate_dirs = []
candidate_dirs += [p for p in KAGGLE_INPUT.glob('**/phase6_daeac_record_splits') if p.is_dir()]
candidate_dirs += [p.parent for p in KAGGLE_INPUT.glob('**/ds1_train.npz')]
candidate_dirs = list(dict.fromkeys(candidate_dirs))
source_data = next((p for p in candidate_dirs if all((p / name).exists() for name in required_npz)), None)
if source_data is None:
    missing_here = [name for name in required_npz if not (data_dest / name).exists()]
    if missing_here:
        raise FileNotFoundError(f'Missing data files: {missing_here}. Attach the Phase 6 record split dataset.')
    print('Using existing data in', data_dest)
else:
    for npz in source_data.glob('*.npz'):
        shutil.copy2(npz, data_dest / npz.name)
    print('Copied data from', source_data, 'to', data_dest)

base_ckpt_dest = REPO_DIR / OUTPUT_DIR / 'checkpoints' / f'{BASE_PREFIX}_best.pt'
base_ckpt_dest.parent.mkdir(parents=True, exist_ok=True)
matches = list(KAGGLE_INPUT.glob(f'**/{BASE_PREFIX}_best.pt'))
if matches and not base_ckpt_dest.exists():
    shutil.copy2(matches[0], base_ckpt_dest)
    print('Copied existing Phase 3 base checkpoint:', matches[0])
print('Base checkpoint path:', base_ckpt_dest)
!ls -lh data/processed/phase6_daeac_record_splits | head

## Verify Phase 3-Only Config

In [ ]:
from scripts.phase6_daeac_paper.common import load_phase1_config
from src.training.daeac_losses import ClassBalancedFocalLoss, build_daeac_classification_loss
import torch

cfg = load_phase1_config(CONFIG)
assert cfg['data']['rr_mode'] == 'real', cfg['data'].get('rr_mode')
assert cfg['losses']['source_cls_loss'] == 'class_balanced_focal', cfg['losses']
rtd = cfg['rtd_daeac']
assert not rtd.get('enabled', False)
assert not rtd['dual_head'].get('enabled', False)
assert not rtd['reliable_pseudo'].get('enabled', False)
assert not rtd['dynamic_weight'].get('enabled', False)
assert not rtd['task_align'].get('enabled', False)
loss = build_daeac_classification_loss(cfg, cfg['data']['num_classes'], class_counts=torch.ones(cfg['data']['num_classes']))
assert isinstance(loss, ClassBalancedFocalLoss), type(loss)
print('OK: Phase 3 only')
print('loss:', cfg['losses'])
print('rtd_daeac:', rtd)

## Verify Real RR Rows

In [ ]:
from torch.utils.data import DataLoader
from src.data.daeac_dataset import DAEACDataset

source_ds = DAEACDataset(
    cfg['data']['source_train'],
    input_key=cfg['data'].get('input_key', 'auto'),
    label_key=cfg['data'].get('label_key', 'y'),
    class_names=cfg['data']['class_names'],
    rr_mode=cfg['data']['rr_mode'],
)
x, y = next(iter(DataLoader(source_ds, batch_size=min(256, len(source_ds)), shuffle=False)))
assert tuple(x.shape[1:]) == (1, 3, 128), tuple(x.shape)
for row, name in [(0, 'morphology'), (1, 'pre_rr_ratio'), (2, 'near_pre_rr_ratio')]:
    values = x[:, 0, row, :]
    print(row, name, 'mean=', float(values.mean()), 'std=', float(values.std()), 'min=', float(values.min()), 'max=', float(values.max()))
assert not torch.allclose(x[:, 0, 1, :], torch.ones_like(x[:, 0, 1, :])), 'Row 1 is neutralized to 1.0'
assert not torch.allclose(x[:, 0, 2, :], torch.ones_like(x[:, 0, 2, :])), 'Row 2 is neutralized to 1.0'
print('OK: real RR rows are present')

## Local Smoke Checks

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_daeac_losses.py', '-q'], check=True)
subprocess.run([sys.executable, 'scripts/phase6_daeac_paper/00_validate_data.py', '--config', CONFIG], check=True)

## One-Epoch Smoke Run

This is intentionally small. It verifies that Phase 3 source pretraining and UDA adaptation can start without full training.

In [ ]:
RUN_SMOKE = True
if RUN_SMOKE:
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/01_train_base.py',
        '--config', CONFIG,
        '--epochs', '1',
        '--max-source-samples', '512',
        '--max-val-samples', '256',
        '--checkpoint-prefix', f'{BASE_PREFIX}_smoke',
    ], check=True)
    smoke_base = REPO_DIR / OUTPUT_DIR / 'checkpoints' / f'{BASE_PREFIX}_smoke_best.pt'
    assert smoke_base.exists(), smoke_base
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/02_adapt_uda.py',
        '--config', CONFIG,
        '--epochs', '1',
        '--max-source-samples', '512',
        '--max-target-samples', '512',
        '--max-val-samples', '256',
        '--init-checkpoint', str(smoke_base),
        '--checkpoint-prefix', f'{ADAPT_PREFIX}_smoke',
    ], check=True)
    print('OK: Phase 3 smoke run completed')

## Full Phase 3 Run

Set the flags below to `True` when you are ready. Full training is intentionally disabled by default.

In [ ]:
RUN_FULL_BASE = False
RUN_FULL_ADAPT = False
PAIRS = ['ds1_ds2']
PAIR_DATA = {
    'ds1_ds2': ('ds1', 'ds2'),
    'ds1_incart': ('ds1', 'incart'),
    'ds1_svdb': ('ds1', 'svdb'),
}

def write_phase3_pair_config(pair, init_checkpoint):
    source, target = PAIR_DATA[pair]
    root = 'data/processed/phase6_daeac_record_splits'
    config_path = REPO_DIR / 'configs' / f'_kaggle_phase3_focal_{pair}.yaml'
    config_text = f'''extends: phase6_daeac_paper_phase3_focal.yaml
data:
  source_train: {root}/{source}_train.npz
  source_eval: {root}/{source}_val.npz
  target_unlabeled: {root}/{target}_train.npz
  target_val: {root}/{target}_val.npz
  target_test: {root}/{target}_test.npz
paths:
  output_dir: outputs/phase6_daeac_paper_phase3_focal_{pair}
adaptation:
  checkpoint_prefix: {ADAPT_PREFIX}_{pair}
  init_checkpoint: {init_checkpoint}
'''
    config_path.write_text(config_text, encoding='utf-8')
    return str(config_path.relative_to(REPO_DIR))

if RUN_FULL_BASE:
    subprocess.run([sys.executable, '-u', 'scripts/phase6_daeac_paper/01_train_base.py', '--config', CONFIG], check=True)

base_ckpt = REPO_DIR / OUTPUT_DIR / 'checkpoints' / f'{BASE_PREFIX}_best.pt'
if RUN_FULL_ADAPT:
    assert base_ckpt.exists(), f'Missing Phase 3 base checkpoint: {base_ckpt}'
    for pair in PAIRS:
        pair_config = write_phase3_pair_config(pair, base_ckpt)
        subprocess.run([
            sys.executable, '-u', 'scripts/phase6_daeac_paper/02_adapt_uda.py',
            '--config', pair_config,
        ], check=True)
else:
    print('Full run disabled. Set RUN_FULL_BASE/RUN_FULL_ADAPT to True when ready.')

## Evaluate And Zip Outputs

In [ ]:
RUN_EVAL = False
EVAL_DATASET = 'target'
if RUN_EVAL:
    for pair in PAIRS:
        pair_config = write_phase3_pair_config(pair, base_ckpt)
        ckpt = Path(f'outputs/phase6_daeac_paper_phase3_focal_{pair}/checkpoints/{ADAPT_PREFIX}_{pair}_best.pt')
        assert ckpt.exists(), ckpt
        subprocess.run([
            sys.executable, '-u', 'scripts/phase6_daeac_paper/03_eval.py',
            '--config', pair_config,
            '--checkpoint', str(ckpt),
            '--method-name', f'{ADAPT_PREFIX}_{pair}_best_v',
            '--dataset', EVAL_DATASET,
        ], check=True)

archive_base = KAGGLE_WORKING / 'phase6_daeac_phase3_focal_outputs'
shutil.make_archive(str(archive_base), 'zip', REPO_DIR / 'outputs')
print('Wrote', archive_base.with_suffix('.zip'))